# Lamination Parameters

Two topics:

1. **Stiffness polars** -- in-plane (Ex, Ey, Gxy) and bending (D11, D66, D16) stiffness vs fibre orientation angle for IM7/8552. Shows which angles maximise specific stiffness components and where D16 (bend-twist coupling) peaks.
2. **Lamination parameter space** -- LP coordinates (xi1A, xi2A) for common layup families plotted inside the Miki (1982) feasibility domain, plus a round-trip reconstruction accuracy check.

Aeroelastic tailoring (D16 optimisation against a washout constraint) is covered in `04_tutorial_aeroelastic_tailoring.ipynb`.

References:
- Gurdal, Haftka & Hajela -- *Design and Optimization of Laminated Composite Materials* (Wiley, 1999) Ch. 4
- Miki (1982) ICCM-4 -- LP feasibility domain

In [ ]:
import sys, os
from pathlib import Path

_nb_dir = Path(os.path.abspath('')).resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from composite_panel import Ply, Laminate, IM7_8552
from composite_panel.lamination_parameters import (
    material_invariants, lamination_parameters, abd_from_lamination_params,
    stiffness_polar, plot_lp_feasibility,
    bend_twist_coupling_index, shear_extension_coupling_index,
)

mat   = IM7_8552()
t_ply = 0.125e-3
h     = 8 * t_ply    # 8-ply symmetric laminate

# Colour palette
bg          = '#0d1117'
panel       = '#161b22'
border      = '#30363d'
text        = '#c9d1d9'
title_color = '#f0f6fc'
blue        = '#58a6ff'
red         = '#f78166'
green       = '#3fb950'
purple      = '#d2a8ff'
orange      = '#ffa657'

def _ax_style(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(panel)
    for spine in ax.spines.values():
        spine.set_color(border)
    ax.tick_params(colors='#8b949e', labelsize=8)
    ax.xaxis.label.set_color(text)
    ax.yaxis.label.set_color(text)
    ax.title.set_color(title_color)
    ax.grid(True, color='#21262d', lw=0.5)
    if title:  ax.set_title(title, fontsize=9)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)

print('Setup complete.')

---
## 1. Stiffness Polars

For a single-angle unidirectional laminate `[theta]_8`, the in-plane moduli (Ex, Ey, Gxy)
and bending stiffnesses (D11, D22, D66, D16) vary strongly with theta.

Key observations to look for:
- Ex peaks at 0 deg, Ey at 90 deg -- highly anisotropic
- Gxy (in-plane shear) peaks near 45 deg
- D16 (bend-twist coupling) peaks near 20-25 deg -- the sweet spot for aeroelastic tailoring
- D16 = 0 at 0 and 90 deg (no coupling at principal angles)

In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(13, 5), facecolor='white')
fig1.subplots_adjust(wspace=0.35, top=0.88)

stiffness_polar(mat, h_total=h, ax_inplane=axes1[0], ax_bending=axes1[1],
                label='IM7/8552', show=False)

# Light theme
for ax in axes1:
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_color('#cccccc')
    ax.tick_params(colors='#333333', labelsize=8)
    ax.xaxis.label.set_color('#333333')
    ax.yaxis.label.set_color('#333333')
    ax.title.set_color('#111111')
    ax.grid(True, color='#e8e8e8', lw=0.5)
    leg = ax.get_legend()
    if leg:
        leg.get_frame().set_facecolor('white')
        leg.get_frame().set_edgecolor('#cccccc')
        for txt in leg.get_texts():
            txt.set_color('#333333')

# Annotate optimal angles on in-plane plot
inv = material_invariants(mat)
thetas = np.linspace(0, 180, 1801)
tr = np.radians(thetas)
u1, u2, u3, u4, u5 = inv['U1'], inv['U2'], inv['U3'], inv['U4'], inv['U5']

a11 = h * (u1 + u2*np.cos(2*tr) + u3*np.cos(4*tr))
a22 = h * (u1 - u2*np.cos(2*tr) + u3*np.cos(4*tr))
a12 = h * (u4 - u3*np.cos(4*tr))
a66 = h * (u5 - u3*np.cos(4*tr))
a16 = h * (0.5*u2*np.sin(2*tr) + u3*np.sin(4*tr))
a26 = h * (0.5*u2*np.sin(2*tr) - u3*np.sin(4*tr))

Ex_arr = np.array([1.0 / (np.linalg.inv(np.array([
    [a11[i], a12[i], a16[i]],
    [a12[i], a22[i], a26[i]],
    [a16[i], a26[i], a66[i]]]))[0, 0] * h) for i in range(len(thetas))])

theta_peak_Ex = thetas[np.argmax(Ex_arr)]
axes1[0].axvline(theta_peak_Ex, color='#555555', lw=0.8, alpha=0.6, ls=':')
axes1[0].text(theta_peak_Ex + 2, axes1[0].get_ylim()[1] * 0.92,
              f'{theta_peak_Ex:.0f}deg', color='#555555', fontsize=7)

D16_arr = (h**3/12) * (0.5*u2*np.sin(2*tr) + u3*np.sin(4*tr))
idx_d16 = np.argmax(np.abs(D16_arr))
axes1[1].axvline(thetas[idx_d16], color='#7c4daa', lw=0.8, alpha=0.6, ls=':')
axes1[1].text(thetas[idx_d16] + 2, axes1[1].get_ylim()[1] * 0.85,
              f'max |D16|: {thetas[idx_d16]:.0f}deg', color='#7c4daa', fontsize=7)

fig1.suptitle('Stiffness Polars  --  IM7/8552  |  Single-angle laminate [theta]_8',
              color='#111111', fontsize=11, y=0.98)

out1 = os.path.join(_repo_root, 'outputs', 'stiffness_polars.png')
fig1.savefig(out1, dpi=150, bbox_inches='tight', facecolor='white')
print(f'Saved -> {out1}')
plt.show()

---
## 2. Lamination Parameter Space

**Lamination parameters** (LPs) are a coordinate transformation that maps any laminate stacking sequence
to a point in a convex feasibility domain. Rather than optimising over discrete ply angles,
you optimise over LP coordinates and then reconstruct a stacking sequence.

The in-plane LPs are:
$$\xi_1^A = \frac{1}{h} \int_{-h/2}^{h/2} \cos(2\theta(z))\, dz, \qquad
\xi_2^A = \frac{1}{h} \int_{-h/2}^{h/2} \cos(4\theta(z))\, dz$$

The Miki (1982) feasibility domain is the parabolic region $2\xi_1^{A2} - 1 \leq \xi_2^A \leq 1$.
Every physically realisable laminate lies inside or on this boundary.

In [ ]:
layups = {
    '[0]_8':             [0,  0,  0,  0,  0,  0,  0,  0],
    '[90]_8':            [90, 90, 90, 90, 90, 90, 90, 90],
    '[+/-45]_2s':        [45, -45, -45, 45, 45, -45, -45, 45],
    '[0/90]_2s':         [0, 90, 90, 0, 0, 90, 90, 0],
    'QI [+/-45/0/90]s':  [45, -45, 0, 90, 90, 0, -45, 45],
    '[30/-30/0/90]s':    [30, -30, 0, 90, 90, 0, -30, 30],
    '[15/-15/0/90]s':    [15, -15, 0, 90, 90, 0, -15, 15],
    '[+/-20/0]s':        [20, -20, 0, 0, 0, 0, -20, 20],
}

print(f"  {'Layup':<22} {'xi1A':>7} {'xi2A':>7} {'xi1D':>7} {'xi2D':>7} "
      f"{'BTC':>7} {'SEC':>7} {'Sym':>5} {'Bal':>5}")
print('  ' + '-' * 78)

lp_data = {}
for name, angles in layups.items():
    plies = [Ply(mat, t_ply, a) for a in angles]
    lam   = Laminate(plies)
    lp    = lamination_parameters(lam)
    btc   = bend_twist_coupling_index(lam)
    sec   = shear_extension_coupling_index(lam)
    lp_data[name] = (lp, btc, sec, lam.is_symmetric, lam.is_balanced)
    print(f"  {name:<22} {lp['xi1A']:>7.3f} {lp['xi2A']:>7.3f} "
          f"{lp['xi1D']:>7.3f} {lp['xi2D']:>7.3f} "
          f"{btc:>7.3f} {sec:>7.3f} "
          f"{'OK' if lam.is_symmetric else 'FAIL':>5} "
          f"{'OK' if lam.is_balanced else 'FAIL':>5}")

In [ ]:
fig2, (ax_lp, ax_btc) = plt.subplots(1, 2, figsize=(11, 5), facecolor=bg)
fig2.subplots_adjust(wspace=0.35, top=0.88)

plot_lp_feasibility(ax=ax_lp, show=False)

colours = [blue, red, green, purple, orange, '#79c0ff', '#ffa198', '#56d364']
for (name, (lp, btc, sec, sym, bal)), col in zip(lp_data.items(), colours):
    ax_lp.scatter(lp['xi1A'], lp['xi2A'], s=70, color=col, zorder=6, label=name)
    ax_lp.annotate(name.split('[')[0] or name[:8],
                   (lp['xi1A'], lp['xi2A']),
                   textcoords='offset points', xytext=(5, -4),
                   color=col, fontsize=6.5)
ax_lp.legend(framealpha=0.2, labelcolor=text, fontsize=6.5, loc='upper right', ncol=1)

# BTC / SEC bar chart
btc_vals = [lp_data[n][1] for n in layups]
sec_vals = [lp_data[n][2] for n in layups]
x = np.arange(len(layups))
width = 0.38
ax_btc.bar(x - width/2, btc_vals, width, label='BTC index', color=purple, alpha=0.8)
ax_btc.bar(x + width/2, sec_vals, width, label='SEC index', color=orange, alpha=0.8)
ax_btc.set_xticks(x)
ax_btc.set_xticklabels(list(layups.keys()), rotation=35, ha='right', fontsize=6.5, color=text)
_ax_style(ax_btc, title='Bend-Twist & Shear-Extension Coupling',
          xlabel='Layup', ylabel='Coupling index [-]')
ax_btc.legend(framealpha=0.2, labelcolor=text, fontsize=8)

fig2.suptitle('Lamination Parameter Space  --  IM7/8552', color=title_color, fontsize=11)

out2 = os.path.join(_repo_root, 'outputs', 'lamination_param_space.png')
fig2.savefig(out2, dpi=150, bbox_inches='tight', facecolor=bg)
print(f'Saved -> {out2}')
plt.show()

---
## 3. LP Reconstruction Accuracy Check

`abd_from_lamination_params()` reconstructs the ABD matrix from LPs. For a discrete laminate
(integer ply angles) this round-trip should be exact — any error is numerical noise.

In [ ]:
print(f"  {'Layup':<22}  {'max A error':>12}  {'max D error':>12}")
print('  ' + '-' * 50)
for name, angles in layups.items():
    plies = [Ply(mat, t_ply, a) for a in angles]
    lam   = Laminate(plies)
    lp    = lamination_parameters(lam)
    A_lp, _, D_lp = abd_from_lamination_params(lam.thickness, lp, mat)
    A_err = np.max(np.abs(A_lp - lam.A)) / (np.max(np.abs(lam.A)) + 1e-30)
    D_err = np.max(np.abs(D_lp - lam.D)) / (np.max(np.abs(lam.D)) + 1e-30)
    print(f"  {name:<22}  {A_err:>12.2e}  {D_err:>12.2e}")

print('\nAll errors are numerical noise — LP reconstruction is exact for discrete laminates.')

---
## 4. Key Takeaways

1. **D16 peaks near 20–25 deg** for IM7/8552 — this is the ply angle that maximises bend-twist coupling. See `04_tutorial_aeroelastic_tailoring.ipynb` for how the optimizer exploits this.
2. **All balanced laminates have BTC = 0.** The ±θ contributions to D16 cancel exactly. Releasing the balance constraint is the prerequisite for engineering D16 ≠ 0.
3. **The LP feasibility domain is convex.** Every physically realisable laminate maps to a point inside the Miki parabola. Convexity makes LP-space optimisation tractable with gradient methods — no combinatorial search over stacking sequences.
4. **Lamination parameters decouple material from thickness.** The invariants U1–U5 are material constants; the LPs encode only the stacking. This separation is useful for multi-material or variable-thickness designs.
5. **LP reconstruction is exact for discrete laminates.** For a continuous LP point that doesn't correspond to a specific stacking, a stacking sequence retrieval (SSR) step is needed to find a manufacturable laminate.